In [2]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

con = duckdb.connect()


In [22]:
# RUTA = "data/transactions/year=2023/**/*.csv"
table_name = "complaints"
RUTA = f"data/{table_name}/**/*.csv"


# Vista sobre todos los archivos: agrega year, month, day y el archivo de origen
con.sql(f"""
    CREATE OR REPLACE VIEW '{table_name}' AS
    SELECT *
    FROM read_csv('{RUTA}',
                  hive_partitioning = true,
                  union_by_name = true,
                  filename = true)
""")


In [23]:

# 1. Estructura: columnas y tipos
print("=== COLUMNAS ===")
con.sql(f"DESCRIBE '{table_name}'").show(max_rows=100)



=== COLUMNAS ===
┌─────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│       column_name       │ column_type │  null   │   key   │ default │  extra  │
│         varchar         │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ complaint_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ creation_date           │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ process_date            │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ case_type               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ category                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ subcategory             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ reception_channel       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │

In [25]:

# 3. Muestra transpuesta, más fácil de leer
print("=== MUESTRA ===")
con.sql(f"""SELECT category, subcategory, reception_channel,
  description, claimed_amount, priority, status,resolution, compensation_granted,
  resolution_satisfaction, is_repeat_complainer AS resume
               FROM {table_name} LIMIT 100""")




=== MUESTRA ===


┌──────────────┬──────────────────────┬───────────────────┬────────────────────────────────────┬────────────────┬──────────┬────────────┬────────────────────────────────────────────────────────────────────────────────────┬──────────────────────┬─────────────────────────┬─────────┐
│   category   │     subcategory      │ reception_channel │            description             │ claimed_amount │ priority │   status   │                                     resolution                                     │ compensation_granted │ resolution_satisfaction │ resume  │
│   varchar    │       varchar        │      varchar      │              varchar               │     double     │ varchar  │  varchar   │                                      varchar                                       │        double        │         double          │ boolean │
├──────────────┼──────────────────────┼───────────────────┼────────────────────────────────────┼────────────────┼──────────┼────────────┼─────────────────

In [26]:
categoricas = ["case_type", "category", "subcategory", "reception_channel",
               "priority", "status", "resolution_satisfaction", "is_repeat_complainer",
               "sla_breached", "currency"]

# Cuántos valores distintos y cuántos nulos tiene cada columna
for col in categoricas:
    con.sql(f"""
        SELECT '{col}' AS columna,
               COUNT(DISTINCT {col}) AS valores_unicos,
               COUNT(*) - COUNT({col}) AS nulos
        FROM {table_name}
    """).show()

# Frecuencia de cada valor
for col in categoricas:
    print(f"\n=== {col} ===")
    con.sql(f"""
        SELECT {col}, COUNT(*) AS n,
               ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
        FROM {table_name}
        GROUP BY {col} ORDER BY n DESC
    """).show(max_rows=50)

┌───────────┬────────────────┬───────┐
│  columna  │ valores_unicos │ nulos │
│  varchar  │     int64      │ int64 │
├───────────┼────────────────┼───────┤
│ case_type │              4 │     0 │
└───────────┴────────────────┴───────┘

┌──────────┬────────────────┬───────┐
│ columna  │ valores_unicos │ nulos │
│ varchar  │     int64      │ int64 │
├──────────┼────────────────┼───────┤
│ category │              5 │     0 │
└──────────┴────────────────┴───────┘

┌─────────────┬────────────────┬───────┐
│   columna   │ valores_unicos │ nulos │
│   varchar   │     int64      │ int64 │
├─────────────┼────────────────┼───────┤
│ subcategory │              5 │  6698 │
└─────────────┴────────────────┴───────┘

┌───────────────────┬────────────────┬───────┐
│      columna      │ valores_unicos │ nulos │
│      varchar      │     int64      │ int64 │
├───────────────────┼────────────────┼───────┤
│ reception_channel │              6 │     0 │
└───────────────────┴────────────────┴───────┘

┌─────

In [27]:
con.sql(f"""SELECT COUNT(*) AS filas, COUNT(DISTINCT complaint_id) AS ids_unicos
            FROM {table_name}""").show()

┌───────┬────────────┐
│ filas │ ids_unicos │
│ int64 │   int64    │
├───────┼────────────┤
│ 67095 │      67095 │
└───────┴────────────┘



In [32]:
#Qué categorías son disputas de transacciones? Define el alcance del agente y las etiquetas de B.
con.sql(f"""
    SELECT category, subcategory, COUNT(*) AS n
    FROM {table_name}
    GROUP BY ALL ORDER BY n DESC
""").show(max_rows=100)

┌──────────────┬──────────────────────┬───────┐
│   category   │     subcategory      │   n   │
│   varchar    │       varchar        │ int64 │
├──────────────┼──────────────────────┼───────┤
│ Transactions │ Cargo no reconocido  │ 12297 │
│ Fees         │ Cobro indebido       │ 12194 │
│ Technical    │ Problema con app     │ 12128 │
│ Branch       │ Atención en sucursal │ 11892 │
│ Service      │ Calidad de servicio  │ 11886 │
│ Branch       │ NULL                 │  1469 │
│ Fees         │ NULL                 │  1359 │
│ Service      │ NULL                 │  1308 │
│ Transactions │ NULL                 │  1283 │
│ Technical    │ NULL                 │  1279 │
└──────────────┴──────────────────────┴───────┘
  10 rows                           3 columns



In [ ]:
# que casos terminan escalados? Qué porcentaje de cada prioridad termina escalado?
con.sql(f"""
    SELECT priority, COUNT(*) AS n,
           ROUND(100.0 * AVG(CASE WHEN status = 'Escalated' THEN 1 ELSE 0 END), 1) AS pct_escalado
    FROM {table_name}
    GROUP BY priority ORDER BY pct_escalado DESC
""").show()
# La prioridad no determina la escalada, aunque los casos de prioridad alta tienen un porcentaje más alto de escalada

┌──────────┬───────┬──────────────┐
│ priority │   n   │ pct_escalado │
│ varchar  │ int64 │    double    │
├──────────┼───────┼──────────────┤
│ Critical │  3355 │          5.5 │
│ Medium   │ 33439 │          5.0 │
│ Low      │ 20411 │          4.9 │
│ High     │  9890 │          4.6 │
└──────────┴───────┴──────────────┘



In [ ]:
# los clientes reicidentes se escalan mas?
con.sql(f"""
    SELECT is_repeat_complainer, COUNT(*) AS n,
           ROUND(100.0 * AVG(CASE WHEN status = 'Escalated' THEN 1 ELSE 0 END), 1) AS pct_escalado
    FROM {table_name} GROUP BY 1
""").show()
# La escalada no depende de si el cliente es reincidente o no

┌──────────────────────┬───────┬──────────────┐
│ is_repeat_complainer │   n   │ pct_escalado │
│       boolean        │ int64 │    double    │
├──────────────────────┼───────┼──────────────┤
│ false                │ 57009 │          5.0 │
│ true                 │ 10086 │          4.9 │
└──────────────────────┴───────┴──────────────┘



In [ ]:
# El canal importa? Qué porcentaje de cada canal termina escalado?
con.sql(f"""
    SELECT reception_channel, COUNT(*) AS n,
           ROUND(100.0 * AVG(CASE WHEN status = 'Escalated' THEN 1 ELSE 0 END), 1) AS pct_escalado,
           ROUND(100.0 * AVG(CASE WHEN TRY_CAST(sla_breached AS BOOLEAN) THEN 1 ELSE 0 END), 1) AS pct_sla_incumplido
    FROM {table_name} GROUP BY 1 ORDER BY pct_escalado DESC
""").show()
# La escalada no depende del canal

┌───────────────────┬───────┬──────────────┬────────────────────┐
│ reception_channel │   n   │ pct_escalado │ pct_sla_incumplido │
│      varchar      │ int64 │    double    │       double       │
├───────────────────┼───────┼──────────────┼────────────────────┤
│ Branch            │  2683 │          5.4 │               19.9 │
│ Regulator         │   717 │          5.4 │               22.5 │
│ App               │  6727 │          5.2 │               20.5 │
│ Email             │ 13323 │          4.9 │               19.8 │
│ Web               │  9884 │          4.9 │               20.4 │
│ Call Center       │ 33761 │          4.9 │               20.1 │
└───────────────────┴───────┴──────────────┴────────────────────┘



In [36]:
#¿Cuánto tarda y cuánto se compensa por categoría
con.sql(f"""
    SELECT category, COUNT(*) AS n,
           ROUND(MEDIAN(TRY_CAST(resolution_days AS INT)), 0) AS dias_mediana,
           ROUND(100.0 * AVG(CASE WHEN TRY_CAST(compensation_granted AS DOUBLE) > 0 THEN 1 ELSE 0 END), 1) AS pct_compensado,
           ROUND(AVG(TRY_CAST(resolution_satisfaction AS INT)), 2) AS satisfaccion
    FROM {table_name} GROUP BY 1 ORDER BY n DESC
""").show()

┌──────────────┬───────┬──────────────┬────────────────┬──────────────┐
│   category   │   n   │ dias_mediana │ pct_compensado │ satisfaccion │
│   varchar    │ int64 │    double    │     double     │    double    │
├──────────────┼───────┼──────────────┼────────────────┼──────────────┤
│ Transactions │ 13580 │         15.0 │            7.3 │         3.09 │
│ Fees         │ 13553 │         16.0 │            7.0 │         2.97 │
│ Technical    │ 13407 │         16.0 │            6.7 │         3.02 │
│ Branch       │ 13361 │         16.0 │            7.0 │         3.05 │
│ Service      │ 13194 │         16.0 │            6.6 │         2.97 │
└──────────────┴───────┴──────────────┴────────────────┴──────────────┘



In [37]:
con.sql(f"""
    SELECT COUNT(*) AS filas,
           COUNT(DISTINCT description) AS textos_distintos,
           ROUND(100.0 * COUNT(DISTINCT description) / COUNT(*), 1) AS pct_distintos
    FROM {table_name}
""").show()

┌───────┬──────────────────┬───────────────┐
│ filas │ textos_distintos │ pct_distintos │
│ int64 │      int64       │    double     │
├───────┼──────────────────┼───────────────┤
│ 67095 │                5 │           0.0 │
└───────┴──────────────────┴───────────────┘

